<a href="https://colab.research.google.com/github/Qureshiii/PyTorch-Learning-Journey/blob/main/04_Building_Blocks_of_Neural_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
# create model class
import torch
import torch.nn as nn


class Model(nn.Module):

  def __init__(self, num_features):

    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(num_features, 3),
        nn.ReLU(),
        nn.Linear(3, 1),
        nn.Sigmoid()
    )

  def forward(self, features):

    out = self.network(features)

    return out

### Create dummy dataset

In [43]:
features = torch.rand(10,5)

### Creating Model

In [52]:
model = Model(features.shape[1])

### Call Model for forward pass

In [53]:
model(features)

tensor([[0.6216],
        [0.6107],
        [0.6020],
        [0.6057],
        [0.6450],
        [0.6374],
        [0.6265],
        [0.6487],
        [0.6181],
        [0.6075]], grad_fn=<SigmoidBackward0>)

In [117]:
!pip install torchinfo

In [57]:
from torchinfo import summary

summary(model, input_size=(10, 5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Sequential: 1-1                        [10, 1]                   --
│    └─Linear: 2-1                       [10, 3]                   18
│    └─ReLU: 2-2                         [10, 3]                   --
│    └─Linear: 2-3                       [10, 1]                   4
│    └─Sigmoid: 2-4                      [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

### Improving the old code

In [118]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [119]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-detection/refs/heads/master/data.csv')

In [120]:
df.drop(columns=['id','Unnamed: 32'],inplace=True)

In [121]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2)

In [130]:
scaler = StandardScaler()

# 1. Train data par fit aur transform dono karein
X_train = scaler.fit_transform(X_train)

# 2. Test data par SIRF transform karein (taaki train wala mean/std hi use ho)
X_test = scaler.transform(X_test)

In [131]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [132]:
X_train_tensors = torch.from_numpy(X_train.astype(np.float32))
X_test_tensors = torch.from_numpy(X_test.astype(np.float32))
y_train_tensors = torch.from_numpy(y_train.astype(np.float32)).view(-1, 1)
y_test_tensors = torch.from_numpy(y_test.astype(np.float32)).view(-1, 1)

In [133]:
import torch.nn as nn
class MySimpleNN(nn.Module):

  def __init__(self, num_features):

    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()


  def forward(self,features):

    out = self.linear(features)
    out = self.sigmoid(out)
    return out

In [134]:
learning_rate = 0.01
epochs = 200

### Built-in Loss function

In [135]:
loss_function = nn.BCELoss()

In [136]:
model = MySimpleNN(X_train_tensors.shape[1])

### Define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
    # 1. Forward pass
    y_pred = model(X_train_tensors)

    # 2. Loss calculation
    loss = loss_function(y_pred, y_train_tensors.view(-1,1))

    # 3. clear grad
    optimizer.zero_grad()

    # 4. Backward pass
    loss.backward()

    # 5. parameter update across ALL layers
    optimizer.step()

    print(f"Epoch : {epoch + 1}, Loss : {loss.item():.4f}")

Epoch : 1, Loss : 0.7944
Epoch : 2, Loss : 0.7703
Epoch : 3, Loss : 0.7475
Epoch : 4, Loss : 0.7259
Epoch : 5, Loss : 0.7055
Epoch : 6, Loss : 0.6862
Epoch : 7, Loss : 0.6679
Epoch : 8, Loss : 0.6506
Epoch : 9, Loss : 0.6342
Epoch : 10, Loss : 0.6187
Epoch : 11, Loss : 0.6041
Epoch : 12, Loss : 0.5902
Epoch : 13, Loss : 0.5770
Epoch : 14, Loss : 0.5646
Epoch : 15, Loss : 0.5528
Epoch : 16, Loss : 0.5416
Epoch : 17, Loss : 0.5310
Epoch : 18, Loss : 0.5209
Epoch : 19, Loss : 0.5113
Epoch : 20, Loss : 0.5021
Epoch : 21, Loss : 0.4935
Epoch : 22, Loss : 0.4852
Epoch : 23, Loss : 0.4773
Epoch : 24, Loss : 0.4697
Epoch : 25, Loss : 0.4625
Epoch : 26, Loss : 0.4557
Epoch : 27, Loss : 0.4491
Epoch : 28, Loss : 0.4427
Epoch : 29, Loss : 0.4367
Epoch : 30, Loss : 0.4309
Epoch : 31, Loss : 0.4253
Epoch : 32, Loss : 0.4199
Epoch : 33, Loss : 0.4147
Epoch : 34, Loss : 0.4097
Epoch : 35, Loss : 0.4049
Epoch : 36, Loss : 0.4003
Epoch : 37, Loss : 0.3958
Epoch : 38, Loss : 0.3915
Epoch : 39, Loss : 0.

In [137]:
model.eval()
with torch.no_grad():
    y_pred = model(X_test_tensors)
    y_pred_class = (y_pred >= 0.5).float()

    # Correct shape comparison: (N, 1) == (N, 1)
    accuracy = (y_pred_class == y_test_tensors).float().mean()
    print(f"\nModel Accuracy : {accuracy.item() * 100:.2f}%")


Model Accuracy : 97.37%
